# Tests et sélection des prompts

## Objectif du notebook

Ce notebook a pour objectif de concevoir, tester et comparer plusieurs prompts destinés à interroger des grands modèles de langage sur les questions médicales MCQU préparées dans le notebook précédent.

Les expériences sont réalisées sur un échantillon reproductible du split `validation`. Le split `test` n’est pas utilisé pendant cette phase afin de le réserver à l’évaluation finale.

Le notebook réalise les étapes suivantes :

1. charger le dataset MCQU préparé ;
2. sélectionner un échantillon de validation reproductible ;
3. définir plusieurs versions de prompts ;
4. construire les messages envoyés aux LLM ;
5. interroger plusieurs modèles dans des conditions identiques ;
6. extraire la lettre et la justification générées ;
7. mesurer la validité du format et l’exactitude des réponses ;
8. sélectionner le prompt retenu pour le benchmark final.

Le prompt sélectionné devra produire une réponse structurée comportant :

- une lettre parmi `A`, `B`, `C`, `D` ou `E` ;
- une justification médicale concise ;
- aucune information extérieure à la question lorsque celle-ci n’est pas nécessaire.

## Pipeline expérimental

```mermaid
flowchart TD
    A["MCQU validation préparé"] --> B["Échantillon reproductible"]
    B --> C["Versions des prompts"]
    C --> D["Appels aux LLM"]
    D --> E["Réponses brutes"]
    E --> F["Extraction lettre et justification"]
    F --> G["Évaluation des prompts"]
    G --> H["Sélection du prompt final"]

# 1. Importations et chemins

In [1]:
from pathlib import Path
import json
import re
import time

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
PROMPT_RESULTS_DIR = RESULTS_DIR / "prompt_tests"

PROMPT_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## 2. Chargement de la validation

In [3]:
VALIDATION_PATH = (
    PROCESSED_DIR
    / "benchmark_mcqu_validation.parquet"
)

df_validation = pd.read_parquet(
    VALIDATION_PATH
)

print("Dimensions :", df_validation.shape)

# affichage des 5 premières lignes du DataFrame
display(
    df_validation[
        [
            "sample_id",
            "medical_subject",
            "question_type",
            "reference_letter",
            "question_context",
        ]
    ].head()
)

Dimensions : (2561, 19)


,sample_id,medical_subject,question_type,reference_letter,question_context
0,mcqu_validation_9940,Ophthalmology,Understanding,C,Question :\nL'amblyopie fonctionnelle se défin...
1,mcqu_validation_14132,Epidemiology,Reasoning,D,"Question :\nDe 1967 à 1972, un groupe de 300 o..."
2,mcqu_validation_23706,Psychiatry,Understanding,E,Question :\n(cochez la réponse fausse) Un synd...
3,mcqu_validation_25123,Physiology,Understanding,E,Question :\n(cochez la réponse fausse) Le diag...
4,mcqu_validation_3365,Rheumatology,Understanding,D,Question :\nAu cours de la détection d'anticor...


In [4]:
print(df_validation.loc[6, "question_context"])

Question :
L'état mixte est un état pathologique marqué par une des 5 propositions suivantes :

Propositions :
A. Intensité des hallucinations
B. Alternance de dépression et d'excitation
C. Coexistence de thèmes dépressifs et de thèmes maniaques
D. Coexistence de traits névrotiques et de traits psychotiques
E. Association d'une dépression et d'une détérioration


## 3. Echantillonage

Pour le pilote initial, cinq questions suffisent. Elles permettent de comparer les trois prompts sans multiplier inutilement les appels. Un pré-benchmark distinct sur 30 questions sera ensuite réalisé avec le prompt sélectionné.

In [14]:
RANDOM_SEED = 42
PROMPT_SAMPLE_SIZE = 5

In [15]:
df_prompt_sample = (
    df_validation.sample(
        n=PROMPT_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
)

In [16]:
display(df_prompt_sample.head())

,sample_id,id,configuration,split,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,choices,choices_text,reference_letter,reference_answer,medical_subject,question_type,task,question_context
2157,mcqu_validation_18257,18257,mcqu,validation,"Une patiente de 26 ans, enceinte de 20 semaine...",Les risques de toxoplasmose congénitale pour l...,90%,70%,50%,20%,"0,5%","{'A': '90%', 'B': '70%', 'C': '50%', 'D': '20%...","A. 90%\nB. 70%\nC. 50%\nD. 20%\nE. 0,5%",D,20%,Gynecology and Obstetrics,Reasoning,QCU,"Cas clinique :\nUne patiente de 26 ans, encein..."
1738,mcqu_validation_7122,7122,mcqu,validation,Vous êtes appelé en urgence en novembre à 8 he...,Ce diagnostic est spécialement fondé sur un ou...,Le caractère collectif de l'intoxication,Le signe de Babinski droit,L'absence de convulsions chez les trois victimes,L'anomalie pupillaire chez la femme,L'absence de cyanose chez les trois victimes,{'A': 'Le caractère collectif de l'intoxicatio...,A. Le caractère collectif de l'intoxication\nB...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning,QCU,Cas clinique :\nVous êtes appelé en urgence en...
1173,mcqu_validation_5451,5451,mcqu,validation,Une femme de 25 ans consulte pour des lésions ...,"Parmi les diagnostics suivants, quel est le pl...",Erythème noueux,Panniculite,Phlébites superficielles,Psoriasis,Erythème polymorphe,"{'A': 'Erythème noueux', 'B': 'Panniculite', '...",A. Erythème noueux\nB. Panniculite\nC. Phlébit...,A,Erythème noueux,Dermatology,Understanding,QCU,Cas clinique :\nUne femme de 25 ans consulte p...
478,mcqu_validation_25802,25802,mcqu,validation,"Madame F.Z. 24 ans, à déjà accouché d'un préma...",(cochez la réponse fausse) Quels sont les exam...,Enregistrement du rythme cardiaque foetal,E.C.B.U.,Bactériologie des pertes vaginales et cervicales,Morphogramme à l'échographie,Localisation placentaire à l'échographie,{'A': 'Enregistrement du rythme cardiaque foet...,A. Enregistrement du rythme cardiaque foetal\n...,D,Morphogramme à l'échographie,Gynecology and Obstetrics,Understanding,QCU,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc..."
1356,mcqu_validation_24634,24634,mcqu,validation,,(cochez la réponse fausse) Devant une fièvre a...,Diarrhée,Dissociation pouls et température,Tuphos,Angine de Vincent,Taches rosées lenticulaires,"{'A': 'Diarrhée', 'B': 'Dissociation pouls et ...",A. Diarrhée\nB. Dissociation pouls et températ...,D,Angine de Vincent,Infectious Diseases,Understanding,QCU,Question :\n(cochez la réponse fausse) Devant ...


Nous allons sauvegarder cet échantillon afin de l'utiliser à l'indentique

In [17]:
SAMPLE_PATH = (
    PROCESSED_DIR
    / "prompt_validation_sample.parquet"
)

df_prompt_sample.to_parquet(
    SAMPLE_PATH,
    index=False
)

print("Échantillon enregistré :", SAMPLE_PATH)

Échantillon enregistré : c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\processed\prompt_validation_sample.parquet


## 4. Création de prompts
Dans cette section, nous allons créer plusieurs prompts des questions à choix multiples. Les prompts seront utilisés pour interroger un modèle de LLM afin d'obtenir une réponse à la question posée. Les prompt seront construits en combinant le contexte de la question, les propositions et une instruction pour le modèle. Nous avons établie 3 prompts différents pour tester les performances du modèle sur la tâche de question à choix unique. Chaque prompt a un niveau de détail croissant, allant d'une simple instruction à une demande d'analyse plus approfondie.


In [8]:
# Prompt à réponse direct (sans justification ni confiance)
PROMPT_V1 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez uniquement avec la lettre correspondante.
""".strip()

In [9]:
# Prompt à réponse avec justification (sans confiance)
PROMPT_V2 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>
""".strip()

In [6]:
# Prompt à réponse avec justification et confiance
PROMPT_V3 = """
Vous devez répondre à une question médicale à choix unique.

{question_context}

Analysez uniquement les informations utiles à la résolution de la question. N’inventez aucune donnée clinique absente du cas présenté.

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>
Confiance : <nombre entier compris entre 0 et 100>
""".strip()

In [10]:
# regroupement des prompts dans un dictionnaire
PROMPT_TEMPLATES = {
    "prompt_v1": PROMPT_V1,
    "prompt_v2": PROMPT_V2,
    "prompt_v3": PROMPT_V3,
}

## 5. Construction des messages

In [11]:
# Fonction pour construire le prompt final en insérant le contexte de la question
def build_prompt(
    question_context,
    prompt_template,
):
    return prompt_template.format(
        question_context=question_context
    )

In [18]:
# test sur un exemple
example_row = df_prompt_sample.iloc[0]

example_prompt = build_prompt(
    question_context=example_row[
        "question_context"
    ],
    prompt_template=PROMPT_V2,
)

print(example_prompt)

Vous devez répondre à une question médicale à choix unique.

Cas clinique :
Une patiente de 26 ans, enceinte de 20 semaines, vient en consultation pour sa visite du 5 ème mois. Le sérodiagnostic de toxoplasmose pratiqué 15 jours auparavant objective un taux d'lgG à 450 U.I., présence d'lgM. Le précédent sérodiagnostic pratiqué en début de grossesse à 6 semaines d'aménorrhée était négatif. Il s'agit d'une séroconversion. Un traitement à la Spiramycine (3 g/j) est instauré

Question :
Les risques de toxoplasmose congénitale pour le foetus sont de l'ordre de :

Propositions :
A. 90%
B. 70%
C. 50%
D. 20%
E. 0,5%

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>


## 6. Tableau expériemental
Nous allons créer un tableau expérimental ayant pour chaque ligne de l'échantillon, les 3 prompts différents à tester.



In [19]:
experiment_rows = []

for _, row in df_prompt_sample.iterrows():
    for prompt_version, template in (
        PROMPT_TEMPLATES.items()
    ):
        experiment_rows.append(
            {
                "sample_id": row["sample_id"],
                "prompt_version": prompt_version,
                "prompt_text": build_prompt(
                    row["question_context"],
                    template,
                ),
                "reference_letter": row[
                    "reference_letter"
                ],
                "reference_answer": row[
                    "reference_answer"
                ],
                "medical_subject": row[
                    "medical_subject"
                ],
                "question_type": row[
                    "question_type"
                ],
            }
        )

df_experiments = pd.DataFrame(
    experiment_rows
)

In [79]:
display(df_experiments.head())

,sample_id,prompt_version,prompt_text,reference_letter,reference_answer,medical_subject,question_type
0,mcqu_validation_18257,prompt_v1,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
1,mcqu_validation_18257,prompt_v2,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
2,mcqu_validation_18257,prompt_v3,Vous devez répondre à une question médicale à ...,D,20%,Gynecology and Obstetrics,Reasoning
3,mcqu_validation_7122,prompt_v1,Vous devez répondre à une question médicale à ...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning
4,mcqu_validation_7122,prompt_v2,Vous devez répondre à une question médicale à ...,A,Le caractère collectif de l'intoxication,Occupational Medicine,Reasoning


## 7. Colonnes à enregistrer

Nous allons retourner certaines informations afin de distinguer : 
- la réponse brute du modèle 
- les informations extraites 
- la réponse attendue 
- le résultat de l’évaluation 
- les erreurs techniques. 


In [20]:

result_columns = [
    "sample_id",
    "model_name",
    "model_version",
    "prompt_version",
    "prompt_text",
    "temperature",
    "raw_response",
    "predicted_letter",
    "generated_justification",
    "declared_confidence",
    "response_format_valid",
    "reference_letter",
    "is_correct",
    "latency_seconds",
    "generation_error",
]

# 8. Extraction des réponses, des justification et des confiances
Avant les appels aux modèles, préparation de la fonction de lecture des réponses :

In [21]:
VALID_LETTERS = {"A", "B", "C", "D", "E"}

In [22]:
def extract_predicted_letter(response):
    if not isinstance(response, str):
        return None

    patterns = [
        r"Réponse\s*:\s*([A-E])",
        r"^\s*([A-E])\s*$",
        r"^\s*([A-E])[\.\)]",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            response,
            flags=re.IGNORECASE,
        )

        if match:
            return match.group(1).upper()

    return None

In [23]:
test_responses = [
    "Réponse : C",
    "Réponse : B\nJustification : ...",
    "A",
    "D. La réponse est...",
    "Je ne sais pas",
    "Confiance = 80"
]

for response in test_responses:
    print(
        response,
        "→",
        extract_predicted_letter(response)
    )

Réponse : C → C
Réponse : B
Justification : ... → B
A → A
D. La réponse est... → D
Je ne sais pas → None
Confiance = 80 → None


In [24]:
# extraction de la justification
def extract_justification(response):
    if not isinstance(response, str):
        return None

    match = re.search(
        r"Justification\s*:\s*(.*?)(?:\nConfiance\s*:|$)",
        response,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        return None

    justification = match.group(1).strip()

    return justification or None

In [25]:
# extraction de la confiance
def extract_confidence(response):
    if not isinstance(response, str):
        return None

    match = re.search(
        r"Confiance\s*[:=]\s*(\d{1,3})",
        response,
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    confidence = int(match.group(1))

    if 0 <= confidence <= 100:
        return confidence

    return None

## 9. Analyse et validation des réponses

Cette fonction regroupe les informations extraites de la réponse brute et vérifie si le modèle a respecté le format demandé par le prompt.

In [26]:
def parse_model_response(
    response,
    prompt_version,
):
    predicted_letter = extract_predicted_letter(
        response
    )

    generated_justification = extract_justification(
        response
    )

    declared_confidence = extract_confidence(
        response
    )

    # Chaque prompt demande un format différent
    if prompt_version == "prompt_v1":
        response_format_valid = (
            predicted_letter is not None
        )

    elif prompt_version == "prompt_v2":
        response_format_valid = (
            predicted_letter is not None
            and generated_justification is not None
        )

    elif prompt_version == "prompt_v3":
        response_format_valid = (
            predicted_letter is not None
            and generated_justification is not None
            and declared_confidence is not None
        )

    else:
        response_format_valid = False

    return {
        "predicted_letter": predicted_letter,
        "generated_justification": (
            generated_justification
        ),
        "declared_confidence": declared_confidence,
        "response_format_valid": (
            response_format_valid
        ),
    }

## 10 Importation d'un LLM et test
Nous allons importer l'API Gemini de Google pour interroger le modèle de language


In [ ]:
import os
from dotenv import load_dotenv
from google import genai


In [ ]:
# Chargement de la clé API du fichier .env

env_path = Path.cwd()/ ".env"

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY is None:
    raise ValueError("GEMINI_API_KEY introuvable")

print("Clé Gemini chargée avec succès.")

Clé Gemini chargée avec succès.


In [29]:
# initialisation client Gemini

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

## 11. Gemini LLM

### Sélection de la famille Gemini Flash

La famille **`Flash`** de Gemini est conçue pour offrir un compromis entre plusieurs critères importants dans le cadre de notre protocole expérimental :

* **capacité de raisonnement**, nécessaire pour répondre à des questions médicales ;
* **rapidité d’exécution**, afin de limiter le temps nécessaire aux expérimentations ;
* **coût d’utilisation**, particulièrement important lors du traitement d'un grand nombre de requêtes ;
* **capacité à traiter un volume important de données**, adaptée à l’évaluation systématique du jeu de données.

Un modèle de la famille **`Pro`** aurait potentiellement permis d’obtenir de meilleures performances, mais au prix d’un **coût plus élevé** et d’un **temps de traitement plus important**.

À l’inverse, un modèle **`Flash-Lite`** aurait permis de réduire davantage les coûts, mais aurait été moins adapté à notre cas d’usage, notamment pour des **questions médicales nécessitant une certaine capacité de raisonnement**.

Le choix de la famille **`Flash`** représente ainsi un compromis pertinent entre **performance, rapidité et coût** pour les expérimentations réalisées dans ce projet.


In [30]:
def call_gemini(
    prompt,
    model_name="gemini-3.6-flash",
):
    start_time = time.perf_counter()

    try:
        interaction = (
            gemini_client.interactions.create(
                model=model_name,
                input=prompt,
                store=False,
            )
        )

        raw_response = interaction.output_text
        generation_error = None

    except Exception as error:
        raw_response = None

        generation_error = (
            f"{type(error).__name__}: {error}"
        )

    latency_seconds = (
        time.perf_counter() - start_time
    )

    return {
        "raw_response": raw_response,
        "latency_seconds": latency_seconds,
        "generation_error": generation_error,
    }

In [31]:

# test du prompt sur un exemple
test_result = call_gemini(
    prompt=example_prompt,
)

print(
    "Réponse brute :\n",
    test_result["raw_response"]
)

print(
    "\nLatence :",
    round(
        test_result["latency_seconds"],
        2,
    ),
    "secondes"
)

print(
    "\nErreur :",
    test_result["generation_error"]
)

Réponse brute :
 Réponse : D
Justification : Le risque de transmission materno-fœtale de la toxoplasmose augmente avec l'âge gestationnel. À 20 semaines d'aménorrhée (2ème trimestre), ce risque de transmission fœtale est d'environ 20 à 30 % (proche de 20 %), alors qu'il est d'environ 10 % au 1er trimestre et atteint 60 à 80 % au 3ème trimestre.

Latence : 6.66 secondes

Erreur : None


In [92]:
# comparaison de la réponse brute avec le format attendu avec la fonction parse_model_response
parsed_test_result = parse_model_response(
    response=test_result["raw_response"],
    prompt_version="prompt_v2",
)

parsed_test_result

{'predicted_letter': 'D',
 'generated_justification': "Le risque de transmission transplacentaire de *Toxoplasma gondii* augmente avec l'âge gestationnel. Au 2ème trimestre (autour de 20 semaines d'aménorrhée), le risque d'infection fœtale (toxoplasmose congénitale) est d'environ 20 à 30 %.",
 'declared_confidence': None,
 'response_format_valid': True}

création d'une fonction afin de faire un test complet sur un exemple de l'échantillon de validation. La fonction suivante `run_single_experiment` réalise tout le pipeline suivant : 

- appel Gemini
- récupération de la réponse
- extraction des informations
- comparaison avec la référence
- création d'une ligne de résultat


In [32]:
def run_single_experiment(
    experiment_row,
    model_name="gemini-3.6-flash",
):
    call_result = call_gemini(
        prompt=experiment_row["prompt_text"],
        model_name=model_name,
    )

    parsed_result = parse_model_response(
        response=call_result["raw_response"],
        prompt_version=experiment_row[
            "prompt_version"
        ],
    )

    predicted_letter = parsed_result[
        "predicted_letter"
    ]

    # Une erreur technique ne doit pas être
    # comptée comme une mauvaise réponse
    if call_result["generation_error"] is not None:
        is_correct = None
    else:
        is_correct = (
            predicted_letter
            == experiment_row["reference_letter"]
        )

    return {
        "sample_id": experiment_row["sample_id"],
        "model_name": "Gemini",
        "model_version": model_name,
        "prompt_version": experiment_row[
            "prompt_version"
        ],
        "prompt_text": experiment_row[
            "prompt_text"
        ],
        "temperature": None,
        "raw_response": call_result[
            "raw_response"
        ],
        "predicted_letter": predicted_letter,
        "generated_justification": (
            parsed_result[
                "generated_justification"
            ]
        ),
        "declared_confidence": (
            parsed_result[
                "declared_confidence"
            ]
        ),
        "response_format_valid": (
            parsed_result[
                "response_format_valid"
            ]
        ),
        "reference_letter": experiment_row[
            "reference_letter"
        ],
        "is_correct": is_correct,
        "latency_seconds": call_result[
            "latency_seconds"
        ],
        "generation_error": call_result[
            "generation_error"
        ],
    }

In [94]:
# test sur une ligne 
first_experiment = df_experiments.iloc[0]

first_result = run_single_experiment(
    experiment_row=first_experiment,
)

first_result

{'sample_id': 'mcqu_validation_18257',
 'model_name': 'Gemini',
 'model_version': 'gemini-3.6-flash',
 'prompt_version': 'prompt_v1',
 'prompt_text': "Vous devez répondre à une question médicale à choix unique.\n\nCas clinique :\nUne patiente de 26 ans, enceinte de 20 semaines, vient en consultation pour sa visite du 5 ème mois. Le sérodiagnostic de toxoplasmose pratiqué 15 jours auparavant objective un taux d'lgG à 450 U.I., présence d'lgM. Le précédent sérodiagnostic pratiqué en début de grossesse à 6 semaines d'aménorrhée était négatif. Il s'agit d'une séroconversion. Un traitement à la Spiramycine (3 g/j) est instauré\n\nQuestion :\nLes risques de toxoplasmose congénitale pour le foetus sont de l'ordre de :\n\nPropositions :\nA. 90%\nB. 70%\nC. 50%\nD. 20%\nE. 0,5%\n\nSélectionnez une seule proposition parmi A, B, C, D ou E.\n\nRépondez uniquement avec la lettre correspondante.",
 'temperature': None,
 'raw_response': 'D',
 'predicted_letter': 'D',
 'generated_justification': None,

Testons les 3 prompts sur la même question

In [95]:
TEST_SAMPLE_ID = "mcqu_validation_18257"

df_same_question = (
    df_experiments[
        df_experiments["sample_id"]
        == TEST_SAMPLE_ID
    ]
    .sort_values("prompt_version")
    .reset_index(drop=True)
)

display(
    df_same_question[
        [
            "sample_id",
            "prompt_version",
            "reference_letter",
        ]
    ]
)

,sample_id,prompt_version,reference_letter
0,mcqu_validation_18257,prompt_v1,D
1,mcqu_validation_18257,prompt_v2,D
2,mcqu_validation_18257,prompt_v3,D


In [96]:
prompt_test_results = []

for _, experiment_row in (
    df_same_question.iterrows()
):
    print(
        "Exécution de",
        experiment_row["prompt_version"],
        "..."
    )

    result = run_single_experiment(
        experiment_row=experiment_row,
        model_name="gemini-3.6-flash",
    )

    prompt_test_results.append(result)

    print(
        "Réponse :",
        result["predicted_letter"],
        "| correcte :",
        result["is_correct"],
        "| format valide :",
        result["response_format_valid"],
        "| latence :",
        round(
            result["latency_seconds"],
            2,
        ),
        "secondes",
    )

Exécution de prompt_v1 ...
Réponse : D | correcte : True | format valide : True | latence : 3.28 secondes
Exécution de prompt_v2 ...
Réponse : D | correcte : True | format valide : True | latence : 5.58 secondes
Exécution de prompt_v3 ...
Réponse : D | correcte : True | format valide : True | latence : 6.2 secondes


In [97]:
df_prompt_test_results = pd.DataFrame(
    prompt_test_results
)

display(
    df_prompt_test_results[
        [
            "sample_id",
            "prompt_version",
            "raw_response",
            "predicted_letter",
            "declared_confidence",
            "response_format_valid",
            "reference_letter",
            "is_correct",
            "latency_seconds",
            "generation_error",
        ]
    ]
)

,sample_id,prompt_version,raw_response,predicted_letter,declared_confidence,response_format_valid,reference_letter,is_correct,latency_seconds,generation_error
0,mcqu_validation_18257,prompt_v1,D,D,NaN,True,D,True,3.276649,None
1,mcqu_validation_18257,prompt_v2,Réponse : D\nJustification : Le risque de tran...,D,NaN,True,D,True,5.581641,None
2,mcqu_validation_18257,prompt_v3,Réponse : D\nJustification : Le risque de tran...,D,95.0,True,D,True,6.197660,None


In [98]:
display(df_prompt_test_results.loc[2,"raw_response"])

"Réponse : D\nJustification : Le risque de transmission materno-fœtale de Toxoplasma gondii augmente progressivement avec l'âge gestationnel. Lors d'une séroconversion survenant au 2ème trimestre (autour de 20 semaines d'aménorrhée), le risque d'infection fœtale (toxoplasmose congénitale) est d'environ 20 à 30 %, la valeur de 20 % correspondante aux données épidémiologiques classiques pour cette période de la grossesse.\nConfiance : 95"

Sélectionnons cinq questions différentes de manière reproductible et leur appliquer les 3 prompts différents afin de générer 15 questions. Celà nous permet d'avoir une cohérence sur le nombre de prompts utilisé d'éviter un nombre inégal d’expériences par prompt.

In [33]:


df_pilot_experiments = (
    df_experiments[
        df_experiments["sample_id"].isin(
            df_prompt_sample["sample_id"]
        )
    ]
    .sort_values(
        [
            "sample_id",
            "prompt_version",
        ]
    )
    .reset_index(drop=True)
)

Générer les 15 appels de test en boucle

In [34]:

pilot_results = []

total_experiments = len(
    df_pilot_experiments
)

for experiment_number, (_, row) in enumerate(
    df_pilot_experiments.iterrows(),
    start=1,
):
    print(
        f"[{experiment_number}/"
        f"{total_experiments}] "
        f"{row['sample_id']} - "
        f"{row['prompt_version']}"
    )

    result = run_single_experiment(
        experiment_row=row,
        model_name="gemini-3.6-flash",
    )

    pilot_results.append(result)

    print(
        "  Réponse :",
        result["predicted_letter"],
        "| correcte :",
        result["is_correct"],
        "| format :",
        result["response_format_valid"],
        "| latence :",
        round(
            result["latency_seconds"],
            2,
        ),
        "s",
    )


[1/15] mcqu_validation_18257 - prompt_v1
  Réponse : D | correcte : True | format : True | latence : 3.13 s
[2/15] mcqu_validation_18257 - prompt_v2
  Réponse : D | correcte : True | format : True | latence : 5.5 s
[3/15] mcqu_validation_18257 - prompt_v3
  Réponse : D | correcte : True | format : True | latence : 5.31 s
[4/15] mcqu_validation_24634 - prompt_v1
  Réponse : D | correcte : True | format : True | latence : 2.76 s
[5/15] mcqu_validation_24634 - prompt_v2
  Réponse : D | correcte : True | format : True | latence : 3.68 s
[6/15] mcqu_validation_24634 - prompt_v3
  Réponse : D | correcte : True | format : True | latence : 3.59 s
[7/15] mcqu_validation_25802 - prompt_v1
  Réponse : D | correcte : True | format : True | latence : 9.98 s
[8/15] mcqu_validation_25802 - prompt_v2
  Réponse : D | correcte : True | format : True | latence : 9.67 s
[9/15] mcqu_validation_25802 - prompt_v3
  Réponse : D | correcte : True | format : True | latence : 8.91 s
[10/15] mcqu_validation_5451 

In [35]:
df_pilot_results = pd.DataFrame(
    pilot_results
)

df_pilot_results = df_pilot_results[
    result_columns
]

print(
    "Dimensions des résultats :",
    df_pilot_results.shape
)

display(
    df_pilot_results[
        [
            "sample_id",
            "prompt_version",
            "predicted_letter",
            "reference_letter",
            "is_correct",
            "response_format_valid",
            "declared_confidence",
            "latency_seconds",
            "generation_error",
        ]
    ]
)

Dimensions des résultats : (15, 15)


,sample_id,prompt_version,predicted_letter,reference_letter,is_correct,response_format_valid,declared_confidence,latency_seconds,generation_error
0,mcqu_validation_18257,prompt_v1,D,D,True,True,NaN,3.128174,None
1,mcqu_validation_18257,prompt_v2,D,D,True,True,NaN,5.495432,None
2,mcqu_validation_18257,prompt_v3,D,D,True,True,95.0,5.313448,None
3,mcqu_validation_24634,prompt_v1,D,D,True,True,NaN,2.764231,None
4,mcqu_validation_24634,prompt_v2,D,D,True,True,NaN,3.682673,None
5,mcqu_validation_24634,prompt_v3,D,D,True,True,100.0,3.592352,None
6,mcqu_validation_25802,prompt_v1,D,D,True,True,NaN,9.983479,None
7,mcqu_validation_25802,prompt_v2,D,D,True,True,NaN,9.668645,None
8,mcqu_validation_25802,prompt_v3,D,D,True,True,95.0,8.912926,None
9,mcqu_validation_5451,prompt_v1,A,A,True,True,NaN,3.368947,None


Enregistrer le test pour analyse

In [36]:
PILOT_RESULTS_PATH = (
    PROMPT_RESULTS_DIR
    / "gemini_prompt_pilot.parquet"
)

df_pilot_results.to_parquet(
    PILOT_RESULTS_PATH,
    index=False,
)

print(
    "Résultats enregistrés dans :",
    PILOT_RESULTS_PATH
)

Résultats enregistrés dans : c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\results\prompt_tests\gemini_prompt_pilot.parquet


## 12. Récupération du parquet et vérification rapide

In [37]:
GEMINI_PROMPT_PILOT = (
    PROMPT_RESULTS_DIR 
    / "gemini_prompt_pilot.parquet"
    )
df_gemini_prompt_pilot_parquet = pd.read_parquet(GEMINI_PROMPT_PILOT)

In [38]:


pilot_summary = (
    df_gemini_prompt_pilot_parquet
    .groupby("prompt_version")
    .agg(
        number_of_experiments=(
            "sample_id",
            "count",
        ),
        accuracy=(
            "is_correct",
            "mean",
        ),
        valid_format_rate=(
            "response_format_valid",
            "mean",
        ),
        mean_latency_seconds=(
            "latency_seconds",
            "mean",
        ),
        technical_errors=(
            "generation_error",
            lambda series: (
                series.notna().sum()
            ),
        ),
    )
    .reset_index()
)

display(pilot_summary)

,prompt_version,number_of_experiments,accuracy,valid_format_rate,mean_latency_seconds,technical_errors
0,prompt_v1,5,1.0,1.0,4.585341,0
1,prompt_v2,5,1.0,1.0,5.364023,0
2,prompt_v3,5,1.0,1.0,5.348188,0


Le cas mcqu_validation_25168 est particulièrement intéressant :

Réponse attendue : B
Réponse produite : A
Confiance déclarée avec V3 : 95 %

Le modèle est donc fortement confiant dans une réponse incorrecte. C’est exactement le type de situation pertinent pour une étude sur les hallucinations et la mauvaise calibration de la confiance. Etudions ce cas.

In [ ]:
sample_id = "mcqu_validation_25168"

question_context = (
    df_prompt_sample.loc[
        df_prompt_sample["sample_id"] == sample_id,
        "question_context",
    ]
    .iloc[0]
)

print("QUESTION ET CONTEXTE\n")
print(question_context)

QUESTION ET CONTEXTE

Question :
(cochez la réponse juste) Au niveau de la jonction neuromusculaire du muscle strié squelettique :

Propositions :
A. Les muscles striés squelettiques sont innervés par des motoneurones myélinisés, de grand calibre, dont le corps cellulaire est situé dans la corne antérieure de la moelle épinière et dans le tronc cérébral
B. Le nombre de fibres musculaires striées squelettiques innervées par un motoneuro- ne est variable
C. Le neurotransmetteur est l'adrénaline
D. Le potentiel d'action (PA) se propage par activation des canaux sodium dont l'ouverture dépend de la libération d'un médiateur intracellulaire libéré par le canal sodique précédent
E. Le curare, comme la toxine botulinique, se fixe aux récepteurs nicotiniques, empêchant ainsi leur activation


In [ ]:
response = (
    df_gemini_prompt_pilot_parquet.loc[
        df_gemini_prompt_pilot_parquet["sample_id"] == sample_id,
        [
            "prompt_version",
            "predicted_letter",
            "reference_letter",
            "generated_justification",
            "declared_confidence",
        ],
    ]
)
display(response)
# justification :
print("\nJustification du LLM")
display(response.iloc[1, response.columns.get_loc("generated_justification")])

,prompt_version,predicted_letter,reference_letter,generated_justification,declared_confidence
9,prompt_v1,A,B,None,NaN
10,prompt_v2,A,B,Les muscles striés squelettiques sont innervés...,NaN
11,prompt_v3,A,B,Les muscles striés squelettiques sont innervés...,95.0



Justification du LLM


"Les muscles striés squelettiques sont innervés par les motoneurones alpha, qui sont des neurones myélinisés de grand calibre (conduction rapide). Leurs corps cellulaires sont situés dans la corne antérieure de la moelle épinière (pour les nerfs spinaux) et dans les noyaux moteurs du tronc cérébral (pour les nerfs crâniens). Les autres propositions sont fausses : le neurotransmetteur est l'acétylcholine (et non l'adrénaline), le potentiel d'action se propage par dépolarisation dépendant du voltage (canaux sodiques voltage-dépendants), et la toxine botulinique bloque la libération présynaptique d'acétylcholine sans se fixer directement sur les récepteurs nicotiniques postsynaptiques comme le fait le curare."

Cette question montre justement pourquoi il ne faut pas assimiler automatiquement une réponse incorrecte à une hallucination.

La référence B est clairement défendable : le nombre de fibres musculaires innervées par un motoneurone varie selon le muscle. Les petites unités motrices contrôlent quelques fibres, tandis que les grandes peuvent en contrôler des centaines ou des milliers. OpenStax

Mais la proposition A est également très plausible :

- les axones moteurs périphériques sont myélinisés ;
- les motoneurones spinaux proviennent de la corne antérieure ;
- les motoneurones des nerfs crâniens moteurs proviennent du tronc cérébral. NCBI Bookshelf

#### Pourquoi MediQAl retient probablement B

La proposition B est incontestablement correcte :

- Le nombre de fibres musculaires innervées par un motoneurone est variable.

La proposition A contient des imprécisions :

- ce sont rigoureusement les axones des motoneurones qui sont myélinisés, pas les corps cellulaires ;
- tous les motoneurones ne sont pas nécessairement « de grand calibre » ;
- la formulation généralise surtout les propriétés des motoneurones alpha.

B est donc probablement la réponse attendue parce qu’elle est plus rigoureuse et sans ambiguïté.

#### Problème dans le raisonnement du LLM

La justification du modèle explique correctement pourquoi A est plausible et pourquoi C, D et E sont fausses. En revanche, elle ne discute absolument pas la proposition B.

| Dimension                            | Conclusion                |
| ------------------------------------ | ------------------------- |
| Réponse selon le benchmark           | Incorrecte                |
| Faits présents dans la justification | Globalement corrects      |
| Erreur de raisonnement               | Oui                       |
| Omission de la proposition B         | Oui                       |
| Confiance mal calibrée               | Oui, 95 % malgré l’erreur |
| Hallucination factuelle nette        | Non ou incertaine         |
| Question potentiellement ambiguë     | Oui                       |

Ce cas sera très intéressant dans le rapport : il montre qu’une évaluation automatique fondée uniquement sur la lettre de référence peut pénaliser une réponse médicalement plausible et surestimer le nombre d’hallucinations

## 12. conservation des erreurs de réponse

Pour une analyse ultérieur, nous allons conserver dans un dataFrame les erreurs, le context, le type de modèle ... Ceci pourra être utilisé pour une analyse et construire un protocole d’annotation qui déterminera le type d'erreur (hallucination, erreur de raisonnement, question ambiguë...)

In [ ]:
# Une seule ligne de contexte par question
df_context = (
    df_prompt_sample[
        [
            "sample_id",
            "question_context",
        ]
    ]
    .drop_duplicates(subset="sample_id")
)

# Sélection des mauvaises réponses
df_wrong_answers = (
    df_gemini_prompt_pilot_parquet[
        df_gemini_prompt_pilot_parquet["is_correct"].eq(False)
    ]
    .merge(
        df_context,
        on="sample_id",
        how="left",
        validate="many_to_one",
    ).reset_index(drop=True)
)

display(df_wrong_answers)

,sample_id,model_name,model_version,prompt_version,prompt_text,temperature,raw_response,predicted_letter,generated_justification,declared_confidence,response_format_valid,reference_letter,is_correct,latency_seconds,generation_error,question_context
0,mcqu_validation_25168,Gemini,gemini-3.6-flash,prompt_v1,Vous devez répondre à une question médicale à ...,None,A,A,None,NaN,True,B,False,8.721004,None,Question :\n(cochez la réponse juste) Au nivea...
1,mcqu_validation_25168,Gemini,gemini-3.6-flash,prompt_v2,Vous devez répondre à une question médicale à ...,None,Réponse : A\n\nJustification : Les muscles str...,A,Les muscles striés squelettiques sont innervés...,NaN,True,B,False,9.603230,None,Question :\n(cochez la réponse juste) Au nivea...
2,mcqu_validation_25168,Gemini,gemini-3.6-flash,prompt_v3,Vous devez répondre à une question médicale à ...,None,Réponse : A\n\nJustification : Les muscles str...,A,Les muscles striés squelettiques sont innervés...,95.0,True,B,False,16.895345,None,Question :\n(cochez la réponse juste) Au nivea...


In [ ]:
output_path = (
    Path("../data/results/prompt_tests")
    / "gemini_test_wrong_answers.parquet"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df_wrong_answers.to_parquet(
    output_path,
    index=False,
)

print(
    f"{len(df_wrong_answers)} mauvaises réponses enregistrées."
)
print(output_path.resolve())

3 mauvaises réponses enregistrées.
C:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\results\prompt_tests\gemini_test_wrong_answers.parquet


## 14. OpenAI LLM

### Sélection du modèle OpenAI

Afin d’évaluer et de comparer plusieurs LLM dans le cadre de ce projet, nous avons choisi d’intégrer un modèle proposé par **OpenAI**. Le modèle sélectionné est **`gpt-5.4-mini-2026-03-17`**.

Ce choix repose sur plusieurs critères :

- **Performances** adaptées aux tâches de type QCM, notamment dans le domaine médical ;
- **Coût réduit** par rapport aux modèles OpenAI de plus grande taille ;
- **Modèle récent**, permettant de bénéficier des dernières améliorations apportées aux modèles de la famille GPT ;
- **Version datée et fixe** (`2026-03-17`), favorisant la stabilité et la reproductibilité des expériences ;
- **Adaptation aux traitements à grand volume**, nécessaire pour évaluer un nombre important de questions.

D’après la documentation officielle d’OpenAI, **GPT-5.4 Mini** est présenté comme un modèle rapide et économique, adapté aux traitements nécessitant un volume important de requêtes.

Son tarif est de :

- **0,75 $ par million de tokens en entrée** ;
- **4,50 $ par million de tokens en sortie**.

Ces caractéristiques constituent un compromis intéressant entre **performances, coût et reproductibilité**, particulièrement adapté au protocole expérimental de ce projet.

In [39]:
from openai import OpenAI
openai_api_key = os.getenv(
    "OPENAI_API_KEY"
)
openai_client = OpenAI(
    api_key=openai_api_key
)

In [40]:
def call_openai(
    prompt,
    model_name="gpt-5.4-mini-2026-03-17",
):
    start_time = time.perf_counter()

    try:
        response = openai_client.responses.create(
            model=model_name,
            input=prompt,
            max_output_tokens=500,
            store=False,
        )

        raw_response = response.output_text
        generation_error = None

    except Exception as error:
        raw_response = None
        generation_error = (
            f"{type(error).__name__}: {error}"
        )

    latency_seconds = (
        time.perf_counter() - start_time
    )

    return {
        "raw_response": raw_response,
        "latency_seconds": latency_seconds,
        "generation_error": generation_error,
    }

In [ ]:

# test du prompt sur un exemple
openai_test_result = call_openai(
    prompt=example_prompt,
)

openai_test_result

print(
    "Réponse brute :\n",
    openai_test_result["raw_response"]
)

print(
    "\nLatence :",
    round(
        openai_test_result["latency_seconds"],
        2,
    ),
    "secondes"
)

print(
    "\nErreur :",
    openai_test_result["generation_error"]
)

Réponse brute :
 Réponse : D  
Justification : Le risque de transmission materno-fœtale de la toxoplasmose augmente avec l’âge gestationnel, mais il reste modéré au 2e trimestre, autour de 20% vers 20 semaines d’aménorrhée. Le traitement par spiramycine vise à réduire ce risque.

Latence : 1.65 secondes

Erreur : None


In [41]:
def run_single_openai_experiment(
    experiment_row,
    model_name="gpt-5.4-mini-2026-03-17",
):
    call_result = call_openai(
        prompt=experiment_row["prompt_text"],
        model_name=model_name,
    )

    parsed_result = parse_model_response(
        response=call_result["raw_response"],
        prompt_version=experiment_row[
            "prompt_version"
        ],
    )

    predicted_letter = parsed_result[
        "predicted_letter"
    ]

    if call_result["generation_error"] is not None:
        is_correct = None
    else:
        is_correct = (
            predicted_letter
            == experiment_row["reference_letter"]
        )

    return {
        "sample_id": experiment_row["sample_id"],
        "model_name": "OpenAI",
        "model_version": model_name,
        "prompt_version": experiment_row[
            "prompt_version"
        ],
        "prompt_text": experiment_row[
            "prompt_text"
        ],
        "temperature": None,
        "raw_response": call_result[
            "raw_response"
        ],
        "predicted_letter": predicted_letter,
        "generated_justification": (
            parsed_result[
                "generated_justification"
            ]
        ),
        "declared_confidence": (
            parsed_result[
                "declared_confidence"
            ]
        ),
        "response_format_valid": (
            parsed_result[
                "response_format_valid"
            ]
        ),
        "reference_letter": experiment_row[
            "reference_letter"
        ],
        "is_correct": is_correct,
        "latency_seconds": call_result[
            "latency_seconds"
        ],
        "generation_error": call_result[
            "generation_error"
        ],
    }

In [ ]:
test_row = df_pilot_experiments.iloc[0]

openai_single_result = (
    run_single_openai_experiment(
        experiment_row=test_row,
    )
)

openai_single_result

{'sample_id': 'mcqu_validation_1199',
 'model_name': 'OpenAI',
 'model_version': 'gpt-5.4-mini-2026-03-17',
 'prompt_version': 'prompt_v1',
 'prompt_text': "Vous devez répondre à une question médicale à choix unique.\n\nQuestion :\nToutes les affections suivantes sauf une peuvent être à l'origine d'uneinsuffisance cardiaque à débit élevé. Laquelle ?\n\nPropositions :\nA. Fistule artério-veineuse\nB. Avitaminose B1\nC. Hyperthyroïdie\nD. Insuffisance mitrale rhumatismale\nE. Anémie sévère\n\nSélectionnez une seule proposition parmi A, B, C, D ou E.\n\nRépondez uniquement avec la lettre correspondante.",
 'temperature': None,
 'raw_response': 'D',
 'predicted_letter': 'D',
 'generated_justification': None,
 'declared_confidence': None,
 'response_format_valid': True,
 'reference_letter': 'D',
 'is_correct': True,
 'latency_seconds': 0.6253316000802442,
 'generation_error': None}

### Lancement du pilot sur OPENAI

Réutilisons exactement `df_pilot_experiments` :

In [42]:
import time

OPENAI_MODEL = (
    "gpt-5.4-mini-2026-03-17"
)
# ajoutons un delai d'attente entre 2 envoies pour éviter les erreurs de rate limit
# pour le test
OPENAI_PAUSE_SECONDS = 1

openai_pilot_results = []

In [ ]:
for experiment_number, (
    _,
    experiment_row,
) in enumerate(
    df_pilot_experiments.iterrows(),
    start=1,
):
    print(
        f"Expérience {experiment_number}"
        f"/{len(df_pilot_experiments)}"
        f" | {experiment_row['sample_id']}"
        f" | {experiment_row['prompt_version']}"
    )

    result = run_single_openai_experiment(
        experiment_row=experiment_row,
        model_name=OPENAI_MODEL,
    )

    openai_pilot_results.append(result)

    print(
        "Prédiction :",
        result["predicted_letter"],
        "| Référence :",
        result["reference_letter"],
        "| Correcte :",
        result["is_correct"],
        "| Latence :",
        round(
            result["latency_seconds"],
            2,
        ),
        "s",
        "| Erreur :",
        result["generation_error"],
    )

    time.sleep(
        OPENAI_PAUSE_SECONDS
    )

Expérience 1/15 | mcqu_validation_1199 | prompt_v1
Prédiction : D | Référence : D | Correcte : True | Latence : 0.8 s | Erreur : None
Expérience 2/15 | mcqu_validation_1199 | prompt_v2
Prédiction : D | Référence : D | Correcte : True | Latence : 1.45 s | Erreur : None
Expérience 3/15 | mcqu_validation_1199 | prompt_v3
Prédiction : D | Référence : D | Correcte : True | Latence : 1.68 s | Erreur : None
Expérience 4/15 | mcqu_validation_12087 | prompt_v1
Prédiction : D | Référence : D | Correcte : True | Latence : 0.86 s | Erreur : None
Expérience 5/15 | mcqu_validation_12087 | prompt_v2
Prédiction : D | Référence : D | Correcte : True | Latence : 1.15 s | Erreur : None
Expérience 6/15 | mcqu_validation_12087 | prompt_v3
Prédiction : D | Référence : D | Correcte : True | Latence : 1.2 s | Erreur : None
Expérience 7/15 | mcqu_validation_21870 | prompt_v1
Prédiction : C | Référence : C | Correcte : True | Latence : 0.7 s | Erreur : None
Expérience 8/15 | mcqu_validation_21870 | prompt_v2
Pr

In [ ]:
# conversion en dataframe et controle 
df_openai_pilot_results = pd.DataFrame(
    openai_pilot_results
)

df_openai_pilot_results = df_openai_pilot_results[
    result_columns
]

print(
    "Dimensions des résultats :",
    df_openai_pilot_results.shape
)

display(
    df_openai_pilot_results[
        [
            "sample_id",
            "prompt_version",
            "predicted_letter",
            "reference_letter",
            "is_correct",
            "response_format_valid",
            "declared_confidence",
            "latency_seconds",
            "generation_error",
        ]
    ]
)

Dimensions des résultats : (15, 15)


,sample_id,prompt_version,predicted_letter,reference_letter,is_correct,response_format_valid,declared_confidence,latency_seconds,generation_error
0,mcqu_validation_1199,prompt_v1,D,D,True,True,NaN,0.799012,None
1,mcqu_validation_1199,prompt_v2,D,D,True,True,NaN,1.451174,None
2,mcqu_validation_1199,prompt_v3,D,D,True,True,96.0,1.683394,None
3,mcqu_validation_12087,prompt_v1,D,D,True,True,NaN,0.859280,None
4,mcqu_validation_12087,prompt_v2,D,D,True,True,NaN,1.151640,None
5,mcqu_validation_12087,prompt_v3,D,D,True,True,98.0,1.200655,None
6,mcqu_validation_21870,prompt_v1,C,C,True,True,NaN,0.697641,None
7,mcqu_validation_21870,prompt_v2,C,C,True,True,NaN,2.132872,None
8,mcqu_validation_21870,prompt_v3,C,C,True,True,87.0,1.893193,None
9,mcqu_validation_25168,prompt_v1,B,B,True,True,NaN,0.996287,None


In [ ]:
PILOT_RESULTS_PATH = (
    PROMPT_RESULTS_DIR
    / "openai_prompt_pilot.parquet"
)

df_openai_pilot_results.to_parquet(
    PILOT_RESULTS_PATH,
    index=False,
)

print(
    "Résultats enregistrés dans :",
    PILOT_RESULTS_PATH
)

Résultats enregistrés dans : c:\Users\MANEL\Dropbox\projet_evaluation_LLM_medicale\data\results\prompt_tests\openai_prompt_pilot.parquet


In [ ]:
OPENAI_PROMPT_PILOT = (
    PROMPT_RESULTS_DIR 
    / "openai_prompt_pilot.parquet"
    )
df_openai_prompt_pilot_parquet = pd.read_parquet(OPENAI_PROMPT_PILOT)

In [ ]:


openai_pilot_summary = (
    df_openai_prompt_pilot_parquet
    .groupby("prompt_version")
    .agg(
        number_of_experiments=(
            "sample_id",
            "count",
        ),
        accuracy=(
            "is_correct",
            "mean",
        ),
        valid_format_rate=(
            "response_format_valid",
            "mean",
        ),
        mean_latency_seconds=(
            "latency_seconds",
            "mean",
        ),
        technical_errors=(
            "generation_error",
            lambda series: (
                series.notna().sum()
            ),
        ),
    )
    .reset_index()
)

display(openai_pilot_summary)

,prompt_version,number_of_experiments,accuracy,valid_format_rate,mean_latency_seconds,technical_errors
0,prompt_v1,5,1.0,1.0,0.870307,0
1,prompt_v2,5,1.0,1.0,1.730055,0
2,prompt_v3,5,1.0,1.0,1.610452,0


## 14. Validation du pilote OpenAI

Le pilote réalisé avec le modèle **`gpt-5.4-mini-2026-03-17`** est validé. Les **15 appels** ont été exécutés avec succès, sans erreur technique, avec **100 % de réponses au format attendu** et **15 réponses correctes sur 15**.

### Comparaison préliminaire des modèles

| Indicateur                      | Gemini 3.6 Flash | GPT-5.4 Mini |
| ------------------------------- | ---------------: | -----------: |
| Expériences                     |               15 |           15 |
| Exactitude                      |             80 % |        100 % |
| Formats valides                 |            100 % |        100 % |
| Erreurs techniques              |                0 |            0 |
| Latence moyenne                 |         ≈ 5,94 s |     ≈ 1,40 s |
| Questions correctement traitées |              4/5 |          5/5 |

Sur ce pilote, **GPT-5.4 Mini a été environ 4,2 fois plus rapide que Gemini 3.6 Flash** en termes de latence moyenne.

Ces résultats restent toutefois **préliminaires**, puisque l’échantillon ne contient que cinq questions évaluées avec trois versions de prompt. Ils ne permettent donc pas de conclure à une supériorité globale d’un modèle. L’objectif principal de cette phase est avant tout de **valider le protocole expérimental et le bon fonctionnement de la chaîne d’évaluation**.

### Exemple de désaccord entre modèles

Un résultat particulièrement intéressant concerne l’échantillon **`mcqu_validation_25168`** :

* **Gemini 3.6 Flash** prédit la réponse `A`, alors que la réponse de référence est `B` ;
* avec `prompt_v3`, Gemini déclare une **confiance de 95 %** malgré cette réponse incorrecte ;
* **GPT-5.4 Mini** prédit correctement `B` avec les trois versions du prompt ;
* avec `prompt_v3`, GPT-5.4 Mini associe à cette réponse une **confiance déclarée de 96 %**.

Cet exemple illustre un cas particulièrement intéressant pour l’étude des hallucinations : **une réponse incorrecte produite avec un niveau de confiance élevé**. Il met également en évidence l’intérêt de comparer plusieurs modèles sur les mêmes questions afin d’identifier leurs désaccords et leurs comportements face à l’incertitude.

### Résultats OpenAI selon la version du prompt

Les performances obtenues avec les trois versions du prompt sont les suivantes :

| Prompt      | Exactitude | Latence moyenne |
| ----------- | ---------: | --------------: |
| `prompt_v1` |      100 % |        ≈ 0,87 s |
| `prompt_v2` |      100 % |        ≈ 1,73 s |
| `prompt_v3` |      100 % |        ≈ 1,61 s |

Le **`prompt_v1`** est logiquement le plus rapide, puisqu’il demande uniquement au modèle de retourner la lettre correspondant à la réponse choisie.

Sur ce petit échantillon, l’ajout d’une **justification** avec `prompt_v2` ou d’un **niveau de confiance** avec `prompt_v3` n’a pas modifié l’exactitude du modèle.

Pour `prompt_v3`, la confiance moyenne déclarée par GPT-5.4 Mini est de :

**95,2 %**

Cependant, toutes les réponses OpenAI étant correctes sur ce pilote, ces résultats ne permettent pas encore d’évaluer correctement la **calibration de la confiance**. Une analyse sur un échantillon plus important, comportant à la fois des réponses correctes et incorrectes, sera nécessaire pour déterminer si le niveau de confiance déclaré par le modèle est réellement associé à sa probabilité de fournir une réponse correcte.


## 14. Réunir les résultats OpenAI et Gemini

In [ ]:
# Rechargement de l'échantillon exact utilisé pour le pilote
df_prompt_sample = pd.read_parquet(SAMPLE_PATH)

assert len(df_prompt_sample) == PROMPT_SAMPLE_SIZE
assert df_prompt_sample["sample_id"].is_unique

Dimensions : (2561, 19)


,sample_id,medical_subject,question_type,reference_letter,question_context
0,mcqu_validation_9940,Ophthalmology,Understanding,C,Question :\nL'amblyopie fonctionnelle se défin...
1,mcqu_validation_14132,Epidemiology,Reasoning,D,"Question :\nDe 1967 à 1972, un groupe de 300 o..."
2,mcqu_validation_23706,Psychiatry,Understanding,E,Question :\n(cochez la réponse fausse) Un synd...
3,mcqu_validation_25123,Physiology,Understanding,E,Question :\n(cochez la réponse fausse) Le diag...
4,mcqu_validation_3365,Rheumatology,Understanding,D,Question :\nAu cours de la détection d'anticor...


In [ ]:
OPENAI_PROMPT_PILOT = (
    PROMPT_RESULTS_DIR 
    / "openai_prompt_pilot.parquet"
    )
df_openai_prompt_pilot_parquet = pd.read_parquet(OPENAI_PROMPT_PILOT)

GEMINI_PROMPT_PILOT = (
    PROMPT_RESULTS_DIR 
    / "gemini_prompt_pilot.parquet"
    )
df_gemini_prompt_pilot_parquet = pd.read_parquet(GEMINI_PROMPT_PILOT)

In [ ]:
df_all_pilot_results = pd.concat(
    [
        df_openai_prompt_pilot_parquet,
        df_gemini_prompt_pilot_parquet,
    ],
    ignore_index=True,
)



In [ ]:
df_model_comparison = (
    df_all_pilot_results
    .groupby(
        [
            "model_name",
            "model_version",
            "prompt_version",
        ]
    )
    .agg(
        number_of_experiments=(
            "sample_id",
            "size",
        ),
        accuracy=(
            "is_correct",
            "mean",
        ),
        valid_format_rate=(
            "response_format_valid",
            "mean",
        ),
        mean_latency_seconds=(
            "latency_seconds",
            "mean",
        ),
        technical_errors=(
            "generation_error",
            lambda values: (
                values.notna().sum()
            ),
        ),
    )
    .reset_index()
)

display(df_model_comparison)

,model_name,model_version,prompt_version,number_of_experiments,accuracy,valid_format_rate,mean_latency_seconds,technical_errors
0,Gemini,gemini-3.6-flash,prompt_v1,5,0.8,1.0,3.751845,0
1,Gemini,gemini-3.6-flash,prompt_v2,5,0.8,1.0,4.713035,0
2,Gemini,gemini-3.6-flash,prompt_v3,5,0.8,1.0,7.068453,0
3,OpenAI,gpt-5.4-mini-2026-03-17,prompt_v1,5,1.0,1.0,0.870307,0
4,OpenAI,gpt-5.4-mini-2026-03-17,prompt_v2,5,1.0,1.0,1.730055,0
5,OpenAI,gpt-5.4-mini-2026-03-17,prompt_v3,5,1.0,1.0,1.610452,0


### Analyse préliminaire des versions de prompts

Les trois versions de prompts obtiennent un **taux de validité de format de 100 %** et ne génèrent **aucune erreur technique** sur l’échantillon pilote étudié.

Concernant les performances, le choix du prompt ne semble pas influencer l’exactitude des réponses sur cet échantillon de cinq questions. **Gemini 3.6 Flash** atteint une exactitude de **80 %** avec chacune des trois versions de prompt, tandis que **GPT-5.4 Mini** obtient **100 %** de réponses correctes dans tous les cas.

Le **`prompt_v1`** présente l’avantage d’être le plus rapide, mais il reste limité pour l’analyse des résultats puisqu’il demande uniquement une lettre en réponse. Il ne permet donc ni d’examiner le raisonnement du modèle ni d’évaluer son niveau de confiance.

Le **`prompt_v3`** apparaît comme le plus riche pour les analyses ultérieures. En plus de la réponse sélectionnée, il fournit une **justification médicale** ainsi qu’un **niveau de confiance déclaré**. Ces informations sont particulièrement utiles pour étudier les phénomènes d’hallucination et identifier les situations dans lesquelles un modèle produit une réponse incorrecte avec un niveau de confiance élevé.

Cette richesse informationnelle s’accompagne toutefois d’un **temps de réponse plus important**, notamment pour Gemini 3.6 Flash, qui doit générer davantage de contenu avant de retourner sa réponse.

Compte tenu de la taille réduite de l’échantillon pilote, ces observations doivent être considérées comme **préliminaires**. Elles permettent principalement de valider le protocole expérimental et d’orienter le choix du prompt pour les expérimentations à plus grande échelle.


## Création d'un comparateur de désaccords

In [ ]:
GEMINI_COMPARISON_COLUMNS = {
    "predicted_letter": "gemini_response",
    "raw_response": "gemini_raw_response",
    "is_correct": "gemini_is_correct",
    "response_format_valid": (
        "gemini_format_valid"
    ),
    "generated_justification": (
        "gemini_justification"
    ),
    "declared_confidence": (
        "gemini_confidence"
    ),
    "generation_error": (
        "gemini_generation_error"
    ),
}


OPENAI_COMPARISON_COLUMNS = {
    "predicted_letter": "openai_response",
    "raw_response": "openai_raw_response",
    "is_correct": "openai_is_correct",
    "response_format_valid": (
        "openai_format_valid"
    ),
    "generated_justification": (
        "openai_justification"
    ),
    "declared_confidence": (
        "openai_confidence"
    ),
    "generation_error": (
        "openai_generation_error"
    ),
}

df_gemini_comparison = (
    df_gemini_prompt_pilot_parquet[
        [
            "sample_id",
            "prompt_version",
            "reference_letter",
            *GEMINI_COMPARISON_COLUMNS.keys(),
        ]
    ]
    .rename(
        columns=GEMINI_COMPARISON_COLUMNS
    )
)

df_openai_comparison = (
    df_openai_prompt_pilot_parquet[
        [
            "sample_id",
            "prompt_version",
            "reference_letter",
            *OPENAI_COMPARISON_COLUMNS.keys(),
        ]
    ]
    .rename(
        columns=OPENAI_COMPARISON_COLUMNS
    )
)

In [ ]:
df_model_comparison_details = (
    df_gemini_comparison.merge(
        df_openai_comparison,
        on=[
            "sample_id",
            "prompt_version",
            "reference_letter",
        ],
        how="inner",
        validate="one_to_one",
    )
)

context_by_sample = (
    df_prompt_sample[
        [
            "sample_id",
            "question_context",
        ]
    ]
    .drop_duplicates(
        subset="sample_id"
    )
)

df_model_comparison_details = (
    df_model_comparison_details.merge(
        context_by_sample,
        on="sample_id",
        how="left",
        validate="many_to_one",
    )
)

In [ ]:
valid_predictions = (
    df_model_comparison_details[
        "gemini_response"
    ].notna()
    &
    df_model_comparison_details[
        "openai_response"
    ].notna()
)

different_predictions = (
    df_model_comparison_details[
        "gemini_response"
    ]
    !=
    df_model_comparison_details[
        "openai_response"
    ]
)

df_model_disagreements = (
    df_model_comparison_details.loc[
        valid_predictions
        & different_predictions
    ]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
DISAGREEMENT_COLUMNS = [
    "sample_id",
    "prompt_version",
    "question_context",
    "reference_letter",

    "gemini_response",
    "gemini_is_correct",
    "gemini_format_valid",
    "gemini_justification",
    "gemini_confidence",

    "openai_response",
    "openai_is_correct",
    "openai_format_valid",
    "openai_justification",
    "openai_confidence",
]

df_model_disagreements = (
    df_model_disagreements[
        DISAGREEMENT_COLUMNS
    ]
)

In [ ]:
columns_to_display = [
    "sample_id",
    "question_context",
    "reference_letter",
    "gemini_response",
    "gemini_justification",
    "openai_response",
    "openai_justification",
]

display(
    df_model_disagreements[
        columns_to_display
    ]
)

,sample_id,prompt_version,question_context,reference_letter,gemini_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,openai_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence
0,mcqu_validation_25168,prompt_v1,Question :\n(cochez la réponse juste) Au nivea...,B,A,False,True,None,NaN,B,True,True,None,NaN
1,mcqu_validation_25168,prompt_v2,Question :\n(cochez la réponse juste) Au nivea...,B,A,False,True,Les muscles striés squelettiques sont innervés...,NaN,B,True,True,Le nombre de fibres musculaires innervées par ...,NaN
2,mcqu_validation_25168,prompt_v3,Question :\n(cochez la réponse juste) Au nivea...,B,A,False,True,Les muscles striés squelettiques sont innervés...,95.0,B,True,True,"Au niveau de la jonction neuromusculaire, une ...",96.0


In [55]:


df_comparison_export = (
    df_model_disagreements[
        DISAGREEMENT_COLUMNS
    ]
    .copy()
)

csv_path = PROMPT_RESULTS_DIR / "model_disagreements_comparison.csv"

df_comparison_export.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",  # bonne compatibilité avec Excel
)

print(
    f"{len(df_comparison_export)} désaccord(s) "
    f"enregistré(s) dans {csv_path}"
)

NameError: name 'df_model_disagreements' is not defined

In [54]:
# fonction pour construire un DataFrame de comparaison entre les modèles
def build_model_comparison(
    df_gemini_results,
    df_openai_results,
    df_context,
):
    """
    Compare les réponses de Gemini et OpenAI
    et retourne :
    - le DataFrame complet de comparaison ;
    - le DataFrame contenant uniquement les désaccords.
    """

    GEMINI_COMPARISON_COLUMNS = {
        "predicted_letter": "gemini_response",
        "raw_response": "gemini_raw_response",
        "is_correct": "gemini_is_correct",
        "response_format_valid": "gemini_format_valid",
        "generated_justification": "gemini_justification",
        "declared_confidence": "gemini_confidence",
        "generation_error": "gemini_generation_error",
    }

    OPENAI_COMPARISON_COLUMNS = {
        "predicted_letter": "openai_response",
        "raw_response": "openai_raw_response",
        "is_correct": "openai_is_correct",
        "response_format_valid": "openai_format_valid",
        "generated_justification": "openai_justification",
        "declared_confidence": "openai_confidence",
        "generation_error": "openai_generation_error",
    }

    # Préparation Gemini
    df_gemini_comparison = (
        df_gemini_results[
            [
                "sample_id",
                "prompt_version",
                "reference_letter",
                *GEMINI_COMPARISON_COLUMNS.keys(),
            ]
        ]
        .rename(columns=GEMINI_COMPARISON_COLUMNS)
    )

    # Préparation OpenAI
    df_openai_comparison = (
        df_openai_results[
            [
                "sample_id",
                "prompt_version",
                "reference_letter",
                *OPENAI_COMPARISON_COLUMNS.keys(),
            ]
        ]
        .rename(columns=OPENAI_COMPARISON_COLUMNS)
    )

    # Fusion Gemini / OpenAI
    df_model_comparison_details = (
        df_gemini_comparison.merge(
            df_openai_comparison,
            on=[
                "sample_id",
                "prompt_version",
                "reference_letter",
            ],
            how="inner",
            validate="one_to_one",
        )
    )

    # Ajout du contexte de la question
    context_by_sample = (
        df_context[
            [
                "sample_id",
                "question_context",
            ]
        ]
        .drop_duplicates(subset="sample_id")
    )

    df_model_comparison_details = (
        df_model_comparison_details.merge(
            context_by_sample,
            on="sample_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Sélection des désaccords
    valid_predictions = (
        df_model_comparison_details["gemini_response"].notna()
        &
        df_model_comparison_details["openai_response"].notna()
    )

    different_predictions = (
        df_model_comparison_details["gemini_response"]
        !=
        df_model_comparison_details["openai_response"]
    )

    df_model_disagreements = (
        df_model_comparison_details.loc[
            valid_predictions
            & different_predictions
        ]
        .copy()
        .reset_index(drop=True)
    )

    disagreement_columns = [
        "sample_id",
        "prompt_version",
        "question_context",
        "reference_letter",

        "gemini_response",
        "gemini_is_correct",
        "gemini_format_valid",
        "gemini_justification",
        "gemini_confidence",

        "openai_response",
        "openai_is_correct",
        "openai_format_valid",
        "openai_justification",
        "openai_confidence",
    ]

    df_model_disagreements = (
        df_model_disagreements[
            disagreement_columns
        ]
    )

    return (
        df_model_comparison_details,
        df_model_disagreements,
    )

In [65]:
# fonction d'enregistrement d'un data frame en fichier csv
def save_csv(
    df,
    output_dir,
    filename,
):
    """
    Enregistre d'un dataframe en csv
    """

    csv_path = output_dir / filename

    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )
    return csv_path

In [61]:
# analyse des retours des modèles et sélection des mauvaises réponses pour analyse
def get_model_errors(
    df_gemini,
    df_openai,
    df_context,
):
    """
    Concatène les résultats Gemini et OpenAI
    et retourne uniquement les mauvaises réponses.
    """

    # Colonnes communes à conserver
    result_columns = [
        "sample_id",
        "reference_letter",
        "predicted_letter",
        "is_correct",
        "generated_justification",
        "declared_confidence",
    ]

    # Préparation Gemini
    df_gemini_errors = (
        df_gemini[result_columns]
        .copy()
    )

    df_gemini_errors["model_name"] = "Gemini"

    # Préparation OpenAI
    df_openai_errors = (
        df_openai[result_columns]
        .copy()
    )

    df_openai_errors["model_name"] = "OpenAI"

    # Concaténation des deux modèles
    df_all_results = pd.concat(
        [
            df_gemini_errors,
            df_openai_errors,
        ],
        ignore_index=True,
    )

    # Ajout du contexte de la question
    context_by_sample = (
        df_context[
            [
                "sample_id",
                "question_context",
            ]
        ]
        .drop_duplicates(
            subset="sample_id"
        )
    )

    df_all_results = (
        df_all_results.merge(
            context_by_sample,
            on="sample_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Sélection uniquement des mauvaises réponses
    df_errors = (
        df_all_results.loc[
            df_all_results["is_correct"] == False
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Renommage pour rendre le tableau plus lisible
    df_errors = df_errors.rename(
        columns={
            "predicted_letter": "model_response",
            "generated_justification": "justification",
            "declared_confidence": "confidence",
        }
    )

    # Ordre final des colonnes
    error_columns = [
        "model_name",
        "sample_id",
        "question_context",
        "reference_letter",
        "model_response",
        "is_correct",
        "justification",
        "confidence",
    ]

    return df_errors[error_columns]

## 15. Sélection du prompt final

Les performances des prompts sont comparées selon leur exactitude, leur respect du format, leur latence et les informations produites.

Le prompt `v1` fournit uniquement une réponse et ne permet pas d’étudier les justifications médicales.  
Le prompt `v2` ajoute une justification.  
Le prompt `v3` ajoute une justification et une confiance déclarée, utiles pour analyser les erreurs produites avec une forte confiance.

Les résultats du pilote restent exploratoires en raison du faible nombre de questions.

Le prompt `v3` est retenu pour la suite du projet. Il fournit une réponse structurée comprenant une lettre, une justification médicale et une confiance déclarée.

La confiance déclarée ne constitue pas une mesure objective de fiabilité. Elle sera utilisée comme une variable expérimentale permettant notamment d’identifier les réponses incorrectes produites avec une confiance élevée.

Cette sélection reste provisoire et doit être confirmée sur un échantillon de validation plus important avant l’évaluation finale sur le split `test`.

In [43]:
SELECTED_PROMPT_VERSION = "prompt_v3"

selected_prompt = PROMPT_TEMPLATES[
    SELECTED_PROMPT_VERSION
]

print(
    "\nContenu du prompt :\n",
    selected_prompt,
)


Contenu du prompt :
 Vous devez répondre à une question médicale à choix unique.

{question_context}

Analysez uniquement les informations utiles à la résolution de la question. N’inventez aucune donnée clinique absente du cas présenté.

Sélectionnez une seule proposition parmi A, B, C, D ou E.

Répondez exactement au format suivant :

Réponse : <lettre>
Justification : <explication médicale concise>
Confiance : <nombre entier compris entre 0 et 100>


## 16. Pré-benchmark sur 30 questions

Le prompt `v3` est maintenant évalué sur 30 nouvelles questions du split `validation`. Les cinq questions du pilote sont exclues. Cette étape vérifie la robustesse du pipeline avant le benchmark final sur le split `test`.

In [44]:
PRE_BENCHMARK_SIZE = 30
PRE_BENCHMARK_RANDOM_SEED = 43

pilot_sample_ids = set(
    df_prompt_sample["sample_id"]
)

df_pre_benchmark_source = df_validation.loc[
    ~df_validation["sample_id"].isin(pilot_sample_ids)
].copy()

if len(df_pre_benchmark_source) < PRE_BENCHMARK_SIZE:
    raise ValueError("Pas assez de questions disponibles.")

df_pre_benchmark_sample = (
    df_pre_benchmark_source.sample(
        n=PRE_BENCHMARK_SIZE,
        random_state=PRE_BENCHMARK_RANDOM_SEED,
    )
    .reset_index(drop=True)
)

PRE_BENCHMARK_SAMPLE_PATH = (
    PROCESSED_DIR / "prompt_v3_prebenchmark_sample.parquet"
)
df_pre_benchmark_sample.to_parquet(
    PRE_BENCHMARK_SAMPLE_PATH,
    index=False,
)

print("Questions du pré-benchmark :", len(df_pre_benchmark_sample))
print("Chevauchement avec le pilote :", len(
    pilot_sample_ids.intersection(df_pre_benchmark_sample["sample_id"])
))

Questions du pré-benchmark : 30
Chevauchement avec le pilote : 0


In [45]:
pre_benchmark_rows = []

for _, row in df_pre_benchmark_sample.iterrows():
    pre_benchmark_rows.append(
        {
            "sample_id": row["sample_id"],
            "prompt_version": SELECTED_PROMPT_VERSION,
            "prompt_text": build_prompt(
                question_context=row["question_context"],
                prompt_template=selected_prompt,
            ),
            "reference_letter": row["reference_letter"],
            "reference_answer": row["reference_answer"],
            "medical_subject": row["medical_subject"],
            "question_type": row["question_type"],
        }
    )

df_pre_benchmark_experiments = pd.DataFrame(pre_benchmark_rows)

assert len(df_pre_benchmark_experiments) == PRE_BENCHMARK_SIZE
assert df_pre_benchmark_experiments["sample_id"].is_unique
assert df_pre_benchmark_experiments["prompt_version"].eq(
    SELECTED_PROMPT_VERSION
).all()

display(df_pre_benchmark_experiments.head())

,sample_id,prompt_version,prompt_text,reference_letter,reference_answer,medical_subject,question_type
0,mcqu_validation_14485,prompt_v3,Vous devez répondre à une question médicale à ...,C,La découverte d'un placenta découronné à l'exa...,Gynecology and Obstetrics,Understanding
1,mcqu_validation_19206,prompt_v3,Vous devez répondre à une question médicale à ...,C,une perception sans objet,Psychiatry,Understanding
2,mcqu_validation_24222,prompt_v3,Vous devez répondre à une question médicale à ...,B,Peut comporter en fonction de l'extension gang...,Hepato-Gastroenterology,Understanding
3,mcqu_validation_6552,prompt_v3,Vous devez répondre à une question médicale à ...,C,Pas de délai légal,Forensic Medicine and Toxicology,Understanding
4,mcqu_validation_1657,prompt_v3,Vous devez répondre à une question médicale à ...,B,Exérèse simple totale,Dermatology,Understanding


### Exécution avec sauvegarde progressive

La fonction suivante sauvegarde le fichier après chaque appel. Si l'exécution est interrompue, relancer la cellule reprend uniquement les expériences manquantes.

In [46]:
def run_experiment_batch(
    experiments,
    runner,
    model_name,
    output_path,
    pause_seconds=1,
):
    if output_path.exists():
        saved_results = pd.read_parquet(output_path)
        results = saved_results.to_dict(orient="records")
        completed_sample_ids = set(saved_results["sample_id"])
    else:
        results = []
        completed_sample_ids = set()

    pending = experiments.loc[
        ~experiments["sample_id"].isin(completed_sample_ids)
    ]

    print("Déjà terminées :", len(completed_sample_ids))
    print("À exécuter :", len(pending))

    for number, (_, experiment_row) in enumerate(
        pending.iterrows(),
        start=1,
    ):
        print(
            f"[{number}/{len(pending)}] "
            f"{experiment_row['sample_id']}"
        )

        result = runner(
            experiment_row=experiment_row,
            model_name=model_name,
        )
        results.append(result)

        pd.DataFrame(results)[result_columns].to_parquet(
            output_path,
            index=False,
        )

        print(
            "Prédiction :", result["predicted_letter"],
            "| Référence :", result["reference_letter"],
            "| Correcte :", result["is_correct"],
            "| Erreur :", result["generation_error"],
        )
        time.sleep(pause_seconds)

    return pd.DataFrame(results)[result_columns]

### Gemini — 30 appels

L'exécution de la cellule suivante effectue des appels API avec le LLM Gémini.

In [47]:
GEMINI_PRE_BENCHMARK_PATH = (
    PROMPT_RESULTS_DIR / "gemini_prompt_v3_prebenchmark.parquet"
)

df_gemini_pre_benchmark = run_experiment_batch(
    experiments=df_pre_benchmark_experiments,
    runner=run_single_experiment,
    model_name="gemini-3.6-flash",
    output_path=GEMINI_PRE_BENCHMARK_PATH,
    pause_seconds=1,
)

Déjà terminées : 0
À exécuter : 30
[1/30] mcqu_validation_14485
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[2/30] mcqu_validation_19206
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[3/30] mcqu_validation_24222
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[4/30] mcqu_validation_6552
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[5/30] mcqu_validation_1657
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[6/30] mcqu_validation_23840
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[7/30] mcqu_validation_14648
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[8/30] mcqu_validation_22662
Prédiction : D | Référence : E | Correcte : False | Erreur : None
[9/30] mcqu_validation_22576
Prédiction : E | Référence : E | Correcte : True | Erreur : None
[10/30] mcqu_validation_22213
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[11/30] mcqu_validation_6

### OpenAI — 30 appels

L'exécution de la cellule suivante effectue des appels API réels.

In [48]:
OPENAI_PRE_BENCHMARK_PATH = (
    PROMPT_RESULTS_DIR / "openai_prompt_v3_prebenchmark.parquet"
)

df_openai_pre_benchmark = run_experiment_batch(
    experiments=df_pre_benchmark_experiments,
    runner=run_single_openai_experiment,
    model_name=OPENAI_MODEL,
    output_path=OPENAI_PRE_BENCHMARK_PATH,
    pause_seconds=OPENAI_PAUSE_SECONDS,
)

Déjà terminées : 0
À exécuter : 30
[1/30] mcqu_validation_14485
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[2/30] mcqu_validation_19206
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[3/30] mcqu_validation_24222
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[4/30] mcqu_validation_6552
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[5/30] mcqu_validation_1657
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[6/30] mcqu_validation_23840
Prédiction : B | Référence : B | Correcte : True | Erreur : None
[7/30] mcqu_validation_14648
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[8/30] mcqu_validation_22662
Prédiction : D | Référence : E | Correcte : False | Erreur : None
[9/30] mcqu_validation_22576
Prédiction : E | Référence : E | Correcte : True | Erreur : None
[10/30] mcqu_validation_22213
Prédiction : C | Référence : C | Correcte : True | Erreur : None
[11/30] mcqu_validation_6

In [49]:
df_pre_benchmark_results = pd.concat(
    [
        df_gemini_pre_benchmark,
        df_openai_pre_benchmark,
    ],
    ignore_index=True,
)

df_pre_benchmark_summary = (
    df_pre_benchmark_results.groupby(
        ["model_name", "model_version"],
        dropna=False,
    )
    .agg(
        experiments=("sample_id", "size"),
        unique_questions=("sample_id", "nunique"),
        correct_answers=("is_correct", lambda values: values.eq(True).sum()),
        accuracy=("is_correct", "mean"),
        valid_format_rate=("response_format_valid", "mean"),
        confidence_available_rate=("declared_confidence", lambda values: values.notna().mean()),
        mean_latency_seconds=("latency_seconds", "mean"),
        technical_errors=("generation_error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

display(df_pre_benchmark_summary)

,model_name,model_version,experiments,unique_questions,correct_answers,accuracy,valid_format_rate,confidence_available_rate,mean_latency_seconds,technical_errors
0,Gemini,gemini-3.6-flash,30,30,28,0.933333,1.0,1.0,4.297675,0
1,OpenAI,gpt-5.4-mini-2026-03-17,30,30,28,0.933333,1.0,1.0,1.406308,0


### Analyse
Analyse des réponses des LLM et des désaccords pour le pré-benchmark

In [52]:
GEMINI_V3_PREBENCHMARK = (
    PROMPT_RESULTS_DIR 
    / "gemini_prompt_v3_prebenchmark.parquet"
    )
df_gemini_v3_prebenchmark_parquet = pd.read_parquet(GEMINI_V3_PREBENCHMARK)

OPENAI_V3_PREBENCHMARK = (
    PROMPT_RESULTS_DIR 
    / "openai_prompt_v3_prebenchmark.parquet"
    )
df_openai_v3_prebenchmark_parquet = pd.read_parquet(OPENAI_V3_PREBENCHMARK)

In [57]:
df_model_comparison_details, df_model_disagreements = (
    build_model_comparison(
        df_gemini_v3_prebenchmark_parquet,
        df_openai_v3_prebenchmark_parquet,
        df_pre_benchmark_source,
    )
)

In [58]:
display(df_model_disagreements)

,sample_id,prompt_version,question_context,reference_letter,gemini_response,gemini_is_correct,gemini_format_valid,gemini_justification,gemini_confidence,openai_response,openai_is_correct,openai_format_valid,openai_justification,openai_confidence
0,mcqu_validation_6461,prompt_v3,Question :\nLe permis d'inhumer peut être déli...,B,B,True,True,Le permis (ou autorisation) d'inhumer est déli...,100,D,False,True,Le permis d’inhumer est délivré par le médecin...,84
1,mcqu_validation_16281,prompt_v3,Cas clinique :\nUne femme jeune de 33 ans est ...,B,D,False,True,Le tableau clinique (insuffance cardiaque fébr...,95,B,True,True,Le tableau évoque une endocardite infectieuse ...,84


In [63]:
df_model_errors = get_model_errors(
    df_gemini_v3_prebenchmark_parquet,
    df_openai_v3_prebenchmark_parquet,
    df_pre_benchmark_source,
)

In [64]:
display(df_model_errors)

,model_name,sample_id,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_validation_22662,Cas clinique :\nVous êtes sollicité au sujet d...,E,D,False,Devant une lacune osseuse de la voûte crânienn...,95
1,Gemini,mcqu_validation_16281,Cas clinique :\nUne femme jeune de 33 ans est ...,B,D,False,Le tableau clinique (insuffance cardiaque fébr...,95
2,OpenAI,mcqu_validation_22662,Cas clinique :\nVous êtes sollicité au sujet d...,E,D,False,Devant une lésion lytique unique de la voûte c...,92
3,OpenAI,mcqu_validation_6461,Question :\nLe permis d'inhumer peut être déli...,B,D,False,Le permis d’inhumer est délivré par le médecin...,84


In [66]:
save_csv(
    df_model_disagreements,
    PROMPT_RESULTS_DIR,
    "model_v3_disagreements_prebenchmark.csv"
)

save_csv(
    df_model_errors,
    PROMPT_RESULTS_DIR,
    "model_v3_errors_prebenchmark.csv"
)

WindowsPath('c:/Users/MANEL/Dropbox/projet_evaluation_LLM_medicale/data/results/prompt_tests/model_v3_errors_prebenchmark.csv')